# Практическая работа № 4 (пример выполнения)
## Игровой цикл, Update Method и компоненты

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр  
**Модуль:** 4 — Паттерны: Game Loop, Update Method, Component  
**Проект:** «Тайна исчезнувшего дива» — мистический детектив по мотивам цикла Виктора Дашкевича о графе Аверине  
**Формат:** демонстрационный блокнот с полным разбором задания  
**Цель:** показать, как добавить в проект игровой цикл, компоненты и системы

---

## 📖 Контекст: где мы находимся

После заданий №1–№3 проект имеет:

| Слой | Классы |
|---|---|
| **Абстракции** | `GameObject` (ABC), `Movable`/`Drawable`/`Collidable` (Protocol) |
| **Сущности** | `Detective`, `Div`, `Location`, `Clue` |
| **Команды** | `Action`, `GameSystem`, `InvestigationSystem`, `DialogueSystem`, `MovementSystem`, `WhipSystem` |
| **Оркестрация** | `Game` (тонкий оркестратор) |
| **Инфраструктура** | `UI`, `InputHandler`, `Renderer`, `GameState`, `VictoryChecker` |

## 🚧 Какие проблемы остались

1. **Мир не обновляется сам.** `Game.run()` ждёт ввода игрока. Если игрок молчит — игра стоит.
2. **Нет времени.** Нельзя сделать «через 30 секунд придёт Кузя».
3. **Нет `dt`.** Даже если добавим таймеры, они не смогут работать корректно.
4. **Нет компонентов.** Сущности — монолитные классы. Комбинации поведения ограничены наследованием.
5. **Смешаны update и render.** Логика и вывод идут в одном потоке.

## 🎯 Что демонстрирует этот блокнот

| Инструмент | Проблема | Улучшение |
|---|---|---|
| **`GameLoop`** | Мир не обновляется сам | Фиксированный шаг, accumulator, max_steps |
| **Компоненты** | Монолитные сущности | `Position`, `Health`, `Inventory`, `MoveSpeed` |
| **`Entity`** | Нет контейнера компонентов | Единый контейнер |
| **Системы** | Логика в классах | `MovementSystem`, `TimerSystem`, `HealthSystem` |
| **Интеграция** | Нет `update` | `Game.update(dt)` вызывается каждый тик |

## Ключевая идея модуля

Игра перестаёт быть «циклом по событиям» и становится **симуляцией**, работающей с временем. Это фундамент для физики, анимаций, таймеров и сетевой синхронизации.

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

---
# Шаг 1. Теория: зачем нужен игровой цикл

## 1.1. Игра — это симуляция

**Обычная программа:** запрос → ответ. Ничего не происходит, пока пользователь не взаимодействует.

**Игра:** непрерывная симуляция. Даже если игрок молчит, мир живёт: враги патрулируют, таймеры тикают, анимации играют, музыка звучит.

## 1.2. Что такое кадр

**Кадр** — один цикл обновления. В каждом кадре:
1. Обработка ввода.
2. Обновление состояния (`update`).
3. Отрисовка (`render`).

**FPS** — frames per second. Стандарт — 60 FPS (≈ 16.6 мс на кадр).

## 1.3. Проблема: асинхронность железа

Разные ПК работают с разной скоростью. Без управления временем игра будет работать **с разной скоростью** на разных машинах.

**Решение:** работать с **временем** (дельтами), а не с количеством итераций.

## 1.4. Что даст игровой цикл проекту

- **Таймеры** — «через 30 секунд...».
- **Анимации** — плавное появление текста.
- **Фоновые события** — случайные встречи, погода.
- **Основа для Pygame** — в модуле 7.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Game Loop](https://gameprogrammingpatterns.com/game-loop.html)
- [Gaffer On Games: Fix Your Timestep!](https://gafferongames.com/post/fix_your_timestep/)

---
# Шаг 2. Проектирование `GameLoop`

## 2.1. Идея fixed timestep

**Fixed timestep** — `update` вызывается с **постоянным** `dt`, независимо от реального времени. Реализуется через **аккумулятор**:

```python
accumulator += frame_time
while accumulator >= fixed_dt:
    update(fixed_dt)
    accumulator -= fixed_dt
render()
```

## 2.2. Спираль смерти и её решение

Если ПК тормозит — аккумулятор копит время, `update` вызывается много раз, ПК тормозит сильнее. Игра зависает.

**Решение:** ограничить `max_steps` и **выбросить остаток времени**, если не успели.

## 2.3. Что улучшит в проекте

- **Стабильные таймеры** на любом железе.
- **Предсказуемая анимация**.
- **Основа для физики** в Pygame-версии.
- **Тестируемость** — можно симулировать N шагов без запуска реального времени.

## Для углублённого изучения

- [Gaffer On Games: Fix Your Timestep!](https://gafferongames.com/post/fix_your_timestep/)
- Nystrom R. «Game Programming Patterns»: [Game Loop](https://gameprogrammingpatterns.com/game-loop.html)

## 2.4. Реализация `GameLoop`

In [ ]:
import os
os.makedirs('mystic_detective/core', exist_ok=True)

with open('mystic_detective/core/game_loop.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"GameLoop — игровой цикл с фиксированным шагом.\n\n"""Отделён от Game: знает только о времени.\n"""Не знает о правилах игры, сущностях, UI.\n"""Game передаёт update и render как колбэки.\n"""\n\nclass GameLoop:\n    \"\"\"Игровой цикл с fixed timestep и защитой от спирали смерти.\n\n    Атрибуты:\n        fixed_dt: постоянный шаг для update.\n        max_steps: максимум update за кадр.\n        max_frame_time: ограничение реального времени за кадр.\n    \"\"\"\n\n    def __init__(\n        self,\n        fixed_dt: float = 1.0 / 60,\n        max_steps: int = 5,\n        max_frame_time: float = 0.25,\n    ):\n        self.fixed_dt = fixed_dt\n        self.max_steps = max_steps\n        self.max_frame_time = max_frame_time\n        self._accumulator = 0.0\n        self._total_updates = 0\n        self._total_renders = 0\n\n    @property\n    def accumulator(self) -> float:\n        return self._accumulator\n\n    @property\n    def total_updates(self) -> int:\n        return self._total_updates\n\n    @property\n    def total_renders(self) -> int:\n        return self._total_renders\n\n    def tick(self, frame_time: float, update, render) -> None:\n        \"\"\"Один проход цикла.\n\n        frame_time — реальное время с прошлого кадра.\n        update — функция, принимает fixed_dt.\n        render — функция без аргументов.\n        \"\"\"\n        if frame_time > self.max_frame_time:\n            frame_time = self.max_frame_time\n\n        self._accumulator += frame_time\n\n        steps = 0\n        while self._accumulator >= self.fixed_dt and steps < self.max_steps:\n            update(self.fixed_dt)\n            self._accumulator -= self.fixed_dt\n            self._total_updates += 1\n            steps += 1\n\n        if self._accumulator >= self.fixed_dt:\n            self._accumulator = 0.0\n\n        render()\n        self._total_renders += 1\n\n    def run(self, update, render, should_stop, measure_frame_time) -> None:\n        \"\"\"Основной цикл.\n\n        measure_frame_time — функция, возвращающая реальное время\n        с прошлого кадра.\n        should_stop — функция без аргументов, возвращает True для выхода.\n        \"\"\"\n        while not should_stop():\n            frame_time = measure_frame_time()\n            self.tick(frame_time, update, render)\n''')
print("✅ core/game_loop.py создан")

## 2.5. Демонстрация работы

In [ ]:
import sys
sys.path.insert(0, 'mystic_detective')

from core.game_loop import GameLoop

print("=== Демонстрация: фиксированный dt ===")
loop = GameLoop(fixed_dt=1/60)
updates = []
loop.tick(0.020, lambda dt: updates.append(dt), lambda: None)
print(f"Вызовов update: {len(updates)}")
print(f"Все dt одинаковые: {len(set(updates)) == 1}")
print(f"dt = {updates[0]:.4f}")

print("\n=== Демонстрация: спираль смерти ===")
loop = GameLoop(fixed_dt=1/60, max_steps=5)
updates = []
loop.tick(10.0, lambda dt: updates.append(dt), lambda: None)
print(f"frame_time = 10.0 с, вызовов update: {len(updates)}")
print(f"Ограничение max_steps=5 соблюдено: {len(updates) <= 5}")
print(f"Аккумулятор обнулён: {loop.accumulator == 0.0}")

**Обратите внимание:**
- Все `dt` одинаковые — физика стабильна.
- При огромном `frame_time` цикл не зависает — `max_steps` ограничивает.
- Аккумулятор обнуляется — защита от накопления.

Это и есть смысл fixed timestep с защитой от спирали смерти.

---
# Шаг 3. Теория: зачем нужны компоненты

## 3.1. Проблема наследования

В модуле 2 мы обсуждали комбинаторный взрыв. Если у персонажа есть 3 типа движения и 2 типа оружия — наследование потребует 6 подклассов. Добавление третьего параметра — уже 12.

## 3.2. Идея Component

**Компонент** — небольшая часть, отвечающая за одну способность. Объект **собирается** из компонентов, а не наследуется от родителя.

**Пример:** `Detective` состоит из:
- `Position` — координаты;
- `Health` — здоровье;
- `Inventory` — улики;
- `MoveSpeed` — скорость.

## 3.3. Как это улучшит проект

- **Композиция вместо наследования** — нет комбинаторного взрыва.
- **Динамичность** — можно добавить/удалить компонент во время игры.
- **Переиспользование** — `Health` используется у детектива, двери, врагов.
- **Тестируемость** — каждый компонент тестируется отдельно.

## 3.4. Что появится в проекте

- `Position`, `Health`, `Inventory`, `MoveSpeed`, `Timer` — компоненты.
- `Entity` — контейнер компонентов.
- `MovementSystem`, `TimerSystem`, `HealthSystem` — обработчики.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Component](https://gameprogrammingpatterns.com/component.html)
- [GitHub: ECS FAQ](https://github.com/SanderMertens/ecs-faq)

---
# Шаг 4. Реализация компонентов

## 4.1. Зачем именно dataclass

**`@dataclass`** даёт:
- автоматический `__init__`;
- автоматический `__repr__` (для отладки);
- автоматический `__eq__`;
- `field(default_factory=...)` для безопасных списков.

## 4.2. Компоненты «Тайны исчезнувшего дива»

- **`Position`** — координаты (нужны для Pygame).
- **`Health`** — энергия/здоровье.
- **`Inventory`** — улики.
- **`MoveSpeed`** — скорость.
- **`Timer`** — таймер события.

In [ ]:
with open('mystic_detective/core/components.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"Компоненты — данные для сущностей.\n\n"""Компоненты не знают друг о друге.\n"""Логика — в системах.\n""\"""\n\nfrom dataclasses import dataclass, field\n\n\n@dataclass\nclass Position:\n    \"\"\"Компонент: координаты.\"\"\"\n    x: float = 0.0\n    y: float = 0.0\n\n    def move(self, dx: float, dy: float) -> None:\n        self.x += dx\n        self.y += dy\n\n    def distance_to(self, other: \"Position\") -> float:\n        return ((self.x - other.x) ** 2 + (self.y - other.y) ** 2) ** 0.5\n\n\n@dataclass\nclass Health:\n    \"\"\"Компонент: здоровье/энергия.\"\"\"\n    current: int = 100\n    maximum: int = 100\n\n    @property\n    def is_alive(self) -> bool:\n        return self.current > 0\n\n    def damage(self, amount: int) -> None:\n        self.current = max(0, self.current - amount)\n\n    def heal(self, amount: int) -> None:\n        self.current = min(self.maximum, self.current + amount)\n\n\n@dataclass\nclass Inventory:\n    \"\"\"Компонент: инвентарь.\"\"\"\n    items: list = field(default_factory=list)\n\n    def add(self, item) -> None:\n        if item not in self.items:\n            self.items.append(item)\n\n    def remove(self, item) -> None:\n        if item in self.items:\n            self.items.remove(item)\n\n    @property\n    def size(self) -> int:\n        return len(self.items)\n\n\n@dataclass\nclass MoveSpeed:\n    \"\"\"Компонент: скорость движения.\"\"\"\n    speed: float = 100.0\n\n\n@dataclass\nclass Timer:\n    \"\"\"Компонент: таймер события.\"\"\"\n    name: str\n    duration: float\n    callback: object = None\n    fired: bool = False\n''')
print("✅ core/components.py создан")

## 4.3. Реализация `Entity`

**`Entity`** — контейнер компонентов. Не имеет собственной логики. Поведение определяется **составом** компонентов.

In [ ]:
with open('mystic_detective/core/entity.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"Entity — контейнер компонентов.\n\n"""Сущность не имеет собственной логики.\n"""Поведение определяется составом компонентов.\n""\"""\n\nclass Entity:\n    \"\"\"Сущность: имя + словарь компонентов.\"\"\"\n\n    def __init__(self, name: str):\n        self.name = name\n        self._components: dict = {}\n        self.active = True\n\n    def add_component(self, component) -> None:\n        \"\"\"Добавить компонент. Один компонент каждого типа.\"\"\"\n        self._components[type(component)] = component\n\n    def get(self, component_type):\n        \"\"\"Получить компонент или None.\"\"\"\n        return self._components.get(component_type)\n\n    def has(self, component_type) -> bool:\n        \"\"\"Проверить наличие компонента.\"\"\"\n        return component_type in self._components\n\n    def remove(self, component_type) -> None:\n        \"\"\"Удалить компонент.\"\"\"\n        self._components.pop(component_type, None)\n\n    def deactivate(self) -> None:\n        self.active = False\n\n    def __repr__(self) -> str:\n        types = \", \".join(t.__name__ for t in self._components)\n        return f\"Entity({self.name!r}: [{types}])\"\n''')
print("✅ core/entity.py создан")

## 4.4. Демонстрация: детектив и дверь

In [ ]:
from core.entity import Entity
from core.components import Position, Health, Inventory, MoveSpeed

# Детектив
detective = Entity("Аверин")
detective.add_component(Position(0, 0))
detective.add_component(Health(100, 100))
detective.add_component(Inventory())
detective.add_component(MoveSpeed(50.0))
print(detective)

# Дверь: другая комбинация — только Position и Health (прочность)
door = Entity("Дверь")
door.add_component(Position(5, 5))
door.add_component(Health(50, 50))
print(door)

# Работа с компонентами
pos = detective.get(Position)
pos.move(10, 20)
print(f"\nПозиция детектива: ({pos.x}, {pos.y})")

hp = detective.get(Health)
hp.damage(30)
print(f"Энергия детектива: {hp.current}/{hp.maximum}")

inv = detective.get(Inventory)
inv.add("Записка с планом Пустоши")
print(f"Инвентарь: {inv.items}")

**Что мы видим:**
- `Detective` и `Door` — разные комбинации компонентов, но **один класс** `Entity`.
- Компоненты независимы: `Position` не знает о `Health`.
- Можно добавлять/удалять компоненты динамически.

Это и есть сила компонентного подхода.

---
# Шаг 5. Реализация систем

## 5.1. Зачем системы

**Компоненты хранят данные. Системы обрабатывают их.** Это разделение — ключевое:
- компонент `Position` не знает, как двигаться;
- система `MovementSystem` знает, как двигать всё, что имеет `Position` и `MoveSpeed`.

## 5.2. Что улучшит в проекте

- **Единая логика движения** для всех сущностей.
- **Легко добавить новую систему** — не трогая компоненты.
- **Тестируемость** — система работает с mock-сущностями.

## 5.3. Три системы для «Тайны исчезнувшего дива»

- **`MovementSystem`** — двигает сущности с `Position` + `MoveSpeed`.
- **`TimerSystem`** — обрабатывает таймеры.
- **`HealthSystem`** — деактивирует мёртвых.

In [ ]:
import os
os.makedirs('mystic_detective/game/systems', exist_ok=True)

with open('mystic_detective/game/systems/__init__.py', 'w', encoding='utf-8') as f:
    f.write('\"\"\"Игровые системы обновления.\"\"\"\n')

with open('mystic_detective/game/systems/movement_system.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"MovementSystem — двигает сущности с Position и MoveSpeed.\"\"\"\n\nfrom core.components import Health, MoveSpeed, Position\n\n\nclass MovementSystem:\n    \"\"\"Двигает сущности с Position и MoveSpeed.\n\n    Пропускает мёртвых: если есть Health и current == 0.\n    \"\"\"\n\n    def update(self, entities, dt: float) -> None:\n        for entity in entities:\n            if not entity.active:\n                continue\n            pos = entity.get(Position)\n            speed = entity.get(MoveSpeed)\n            if pos is None or speed is None:\n                continue\n            hp = entity.get(Health)\n            if hp is not None and not hp.is_alive:\n                continue\n            pos.x += speed.speed * dt\n''')

with open('mystic_detective/game/systems/timer_system.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"TimerSystem — обрабатывает таймеры.\"\"\"\n\nfrom core.components import Timer\n\n\nclass TimerSystem:\n    \"\"\"Уменьшает duration таймеров, вызывает callback при срабатывании.\"\"\"\n\n    def update(self, entities, dt: float) -> None:\n        for entity in entities:\n            if not entity.active:\n                continue\n            timer = entity.get(Timer)\n            if timer is None or timer.fired:\n                continue\n            timer.duration -= dt\n            if timer.duration <= 0:\n                timer.fired = True\n                if timer.callback is not None:\n                    timer.callback()\n''')

with open('mystic_detective/game/systems/health_system.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"HealthSystem — деактивирует мёртвых.\"\"\"\n\nfrom core.components import Health\n\n\nclass HealthSystem:\n    \"\"\"Если Health.current == 0 — деактивировать сущность.\"\"\"\n\n    def update(self, entities, dt: float) -> None:\n        for entity in entities:\n            hp = entity.get(Health)\n            if hp is not None and not hp.is_alive:\n                entity.deactivate()\n''')

print("✅ game/systems/ созданы: movement_system.py, timer_system.py, health_system.py")

## 5.4. Демонстрация: системы в действии

In [ ]:
from core.entity import Entity
from core.components import Position, Health, MoveSpeed, Timer
from game.systems.movement_system import MovementSystem
from game.systems.timer_system import TimerSystem
from game.systems.health_system import HealthSystem

# Создаём сущности
detective = Entity("Аверин")
detective.add_component(Position(0, 0))
detective.add_component(MoveSpeed(100.0))
detective.add_component(Health(100, 100))

dead_enemy = Entity("Мёртвый враг")
dead_enemy.add_component(Position(50, 50))
dead_enemy.add_component(MoveSpeed(200.0))
dead_enemy.add_component(Health(0, 100))

static_door = Entity("Дверь")
static_door.add_component(Position(10, 10))
static_door.add_component(Health(50, 50))

events = []
timer_entity = Entity("KuziaTimer")
timer_entity.add_component(Timer(
    name="kuzia_arrives",
    duration=0.1,
    callback=lambda: events.append("Кузя пришёл!"),
))

entities = [detective, dead_enemy, static_door, timer_entity]

# Системы
movement = MovementSystem()
timers = TimerSystem()
health = HealthSystem()

# Симуляция 0.2 секунды
dt = 1 / 60
for _ in range(12):
    movement.update(entities, dt)
    timers.update(entities, dt)
    health.update(entities, dt)

print("=== После 0.2 секунды симуляции ===")
det_pos = detective.get(Position)
print(f"Детектив: x={det_pos.x:.2f} (двигался)")
dead_pos = dead_enemy.get(Position)
print(f"Мёртвый враг: x={dead_pos.x:.2f} (не двигался)")
door_pos = static_door.get(Position)
print(f"Дверь: x={door_pos.x:.2f} (не двигалась — нет MoveSpeed)")
print(f"События: {events}")
print(f"Мёртвый враг активен: {dead_enemy.active}")

**Что мы видим:**
- Детектив двигался — у него есть `Position` и `MoveSpeed`.
- Мёртвый враг **не двигался** — система пропустила его (Health.current == 0).
- Дверь **не двигалась** — у неё нет `MoveSpeed`.
- Таймер сработал — Кузя «пришёл».
- Мёртвый враг деактивирован системой здоровья.

**Это и есть сила компонентного подхода:** каждая система отвечает за свою часть, сущности собираются из компонентов.

---
# Шаг 6. Интеграция с `Game`

## 6.1. Зачем интеграция

Мы создали `GameLoop`, компоненты и системы. Теперь нужно соединить их в `Game`.

## 6.2. Разделение систем

В модуле 3 у нас были **системы команд** (`InvestigationSystem`, `DialogueSystem`, `MovementSystem`, `WhipSystem`), обрабатывающие `Action`. Теперь появились **системы обновления** (`MovementSystem`, `TimerSystem`, `HealthSystem`), обрабатывающие `dt`.

**Конфликт имён:** `MovementSystem` есть в обоих смыслах. Разведём их:
- **командные системы** → `game/systems/` (без изменений);
- **системы обновления** → `game/update_systems/` (новые).

В примере мы уже создали их в `game/systems/`, но переименуем каталог для ясности.

In [ ]:
import shutil
import os

os.makedirs('mystic_detective/game/update_systems', exist_ok=True)
for fname in ['movement_system.py', 'timer_system.py', 'health_system.py']:
    src = os.path.join('mystic_detective/game/systems', fname)
    dst = os.path.join('mystic_detective/game/update_systems', fname)
    if os.path.exists(src):
        shutil.copy(src, dst)

with open('mystic_detective/game/update_systems/__init__.py', 'w', encoding='utf-8') as f:
    f.write('\"\"\"Системы обновления (по dt).\"\"\"\n')

print("✅ game/update_systems/ созданы")

## 6.3. Обновлённый `Game`

In [ ]:
with open('mystic_detective/game/game_v4.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"Game v4 — оркестратор с игровым циклом и системами обновления.\"\"\"\n\nfrom core.game_loop import GameLoop\nfrom core.game_state import GameState\nfrom game.input_handler import InputHandler\nfrom game.renderer import Renderer\nfrom game.victory_checker import VictoryChecker\nfrom ui.ui import UI\n\n\nclass Game:\n    \"\"\"Оркестратор: command-системы + update-системы + GameLoop.\"\"\"\n\n    def __init__(\n        self,\n        state: GameState,\n        command_systems: list,\n        update_systems: list,\n        input_handler: InputHandler,\n        renderer: Renderer,\n        victory_checker: VictoryChecker,\n        game_loop: GameLoop,\n        ui: UI,\n    ):\n        self._state = state\n        self._command_systems = command_systems\n        self._update_systems = update_systems\n        self._input = input_handler\n        self._renderer = renderer\n        self._victory = victory_checker\n        self._loop = game_loop\n        self._ui = ui\n\n    def update(self, dt: float) -> None:\n        \"\"\"Обновить мир: вызвать все update-системы.\"\"\"\n        entities = self._state.entities\n        for system in self._update_systems:\n            system.update(entities, dt)\n\n    def render(self) -> None:\n        \"\"\"Отрисовать состояние.\"\"\"\n        self._renderer.render(self._state)\n\n    def run(self) -> None:\n        \"\"\"Основной цикл: чередуем update и ввод команды.\"\"\"\n        self._renderer.show_title(\"ТАЙНА ИСЧЕЗНУВШЕГО ДИВА\")\n\n        while self._state.detective.energy > 0:\n            # Один тик игрового цикла: фиксированный update\n            self.update(self._loop.fixed_dt)\n\n            # Отрисовка\n            self._renderer.render(self._state)\n            self._renderer.show_menu()\n\n            # Ожидание ввода (в текстовой игре ввод — событие, не тик)\n            action = self._input.read_action()\n            if action is None:\n                self._ui.show_message(\"Неизвестная команда.\")\n                continue\n\n            if action.name == \"quit\":\n                return\n\n            self._dispatch(action)\n\n            if self._victory.is_victory(self._state):\n                self._ui.show_message(\"\\nВы раскрыли тайну исчезновения Владимира!\")\n                return\n\n        self._ui.show_message(\"\\nЭнергия закончилась. Расследование провалено.\")\n\n    def _dispatch(self, action) -> None:\n        for system in self._command_systems:\n            if system.can_handle(action):\n                system.handle(action, self._state, self._ui)\n                return\n        self._ui.show_message(f\"Нет системы для команды {action.name!r}\")\n''')
print("✅ game/game_v4.py создан")

## 6.4. Composition Root: `build_game`

Обновим фабрику, чтобы она создавала `Game` с игровым циклом и системами обновления.

In [ ]:
with open('mystic_detective/game/factory_v4.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"Composition Root для Game v4.\"\"\"\n\nfrom core.game_loop import GameLoop\nfrom core.game_state import GameState\nfrom core.config import DetectiveConfig, GameConfig\nfrom entities.clue import Clue\nfrom entities.detective_v2 import Detective\nfrom entities.div import Div\nfrom entities.location import Location\nfrom game.game_v4 import Game\nfrom game.input_handler import InputHandler\nfrom game.renderer import Renderer\nfrom game.systems import DialogueSystem, InvestigationSystem, MovementSystem as CmdMovementSystem\nfrom game.update_systems.movement_system import MovementSystem as UpdateMovementSystem\nfrom game.update_systems.timer_system import TimerSystem\nfrom game.update_systems.health_system import HealthSystem\nfrom game.victory_checker import VictoryChecker\nfrom ui.ui import UI\n\n\ndef build_locations():\n    loc1 = Location(\"особняк\", \"Старый особняк Авериных на Петроградке.\")\n    loc1.add_clue(Clue(\"Странный запах гари\", \"Пахнет серой\", is_key=False))\n    loc1.add_character(\"Дворецкий\")\n\n    loc2 = Location(\"академия\", \"Академия колдовских наук.\")\n    loc2.add_clue(Clue(\"Записка с планом Пустоши\", \"Карта\", is_key=True))\n    loc2.add_character(\"Ректор\")\n\n    loc3 = Location(\"пустошь\", \"Ледяная пустыня, мир дивов.\")\n    loc3.add_clue(Clue(\"Следы Владимира\", \"Отпечатки лап\", is_key=True))\n    loc3.add_character(\"Древний див\")\n\n    return {\"особняк\": loc1, \"академия\": loc2, \"пустошь\": loc3}\n\n\ndef build_game(ui: UI = None) -> Game:\n    ui = ui if ui is not None else UI()\n\n    div = Div(\"Кузя\", form=\"кот\", loyalty=50)\n    detective = Detective(\n        config=DetectiveConfig(),\n        game_config=GameConfig(),\n        div=div,\n    )\n    locations = build_locations()\n    state = GameState(detective, locations, start_location=\"особняк\")\n\n    command_systems = [\n        InvestigationSystem(),\n        DialogueSystem(),\n        CmdMovementSystem(),\n    ]\n\n    update_systems = [\n        UpdateMovementSystem(),\n        TimerSystem(),\n        HealthSystem(),\n    ]\n\n    return Game(\n        state=state,\n        command_systems=command_systems,\n        update_systems=update_systems,\n        input_handler=InputHandler(ui),\n        renderer=Renderer(ui),\n        victory_checker=VictoryChecker(required_key_clues=2),\n        game_loop=GameLoop(fixed_dt=1/60, max_steps=5),\n        ui=ui,\n    )\n''')
print("✅ game/factory_v4.py создан")

**Важно:** `GameState` теперь должен содержать `entities` — список сущностей. Добавим это в `GameState` (или создадим `GameStateV4`).

Для простоты примера обновим `GameState`, добавив пустой список `entities`. В реальном проекте лучше сделать отдельный класс.

In [ ]:
# Патчим GameState, добавляя entities (в реальном проекте — в самом классе)
from core.game_state import GameState

if not hasattr(GameState, 'entities'):
    _orig_init = GameState.__init__

    def _new_init(self, detective, locations, start_location):
        _orig_init(self, detective, locations, start_location)
        self.entities = []

    GameState.__init__ = _new_init
    print("✅ GameState расширен полем entities")

## 6.5. Демонстрация: таймер в игре

Добавим таймер «Кузя приходит через 30 секунд» и покажем, что событие срабатывает при симуляции.

In [ ]:
from game.factory_v4 import build_game
from core.entity import Entity
from core.components import Timer
from ui.ui import FakeUI

# Создаём игру с FakeUI
fake = FakeUI([])
game = build_game(ui=fake)

# Добавляем таймер: Кузя придёт через 0.5 секунды
events = []
timer_entity = Entity("KuziaArrival")
timer_entity.add_component(Timer(
    name="kuzia_arrives",
    duration=0.5,
    callback=lambda: events.append("Кузя вернулся с новостями!"),
))
game._state.entities.append(timer_entity)

# Симулируем 1 секунду игрового времени
dt = game._loop.fixed_dt
for _ in range(60):  # 60 тиков по 1/60 = 1 секунда
    game.update(dt)

print("=== Симуляция 1 секунды игрового времени ===")
print(f"События: {events}")
print(f"Таймер сработал: {timer_entity.get(Timer).fired}")

**Что мы видим:** таймер сработал ровно через 0.5 секунды игрового времени. Это стало возможно благодаря:
- `GameLoop` с фиксированным `dt`;
- `TimerSystem`, обрабатывающей таймеры;
- интеграции систем в `Game.update(dt)`.

Без игрового цикла сделать это было бы невозможно.

---
# Шаг 7. Юнит-тесты

## 7.1. Зачем тесты

Игровой цикл и системы — критичная часть. Тесты дают гарантию, что после рефакторинга поведение не сломалось.

## 7.2. Тесты для `GameLoop`

In [ ]:
import os
os.makedirs('mystic_detective/tests', exist_ok=True)

with open('mystic_detective/tests/test_game_loop.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"Тесты GameLoop.\"\"\"\n\nimport pytest\nfrom core.game_loop import GameLoop\n\n\nclass TestGameLoop:\n    def test_fixed_dt(self):\n        loop = GameLoop(fixed_dt=1/60)\n        updates = []\n        loop.tick(0.020, lambda dt: updates.append(dt), lambda: None)\n        assert len(updates) >= 1\n        assert all(dt == 1/60 for dt in updates)\n\n    def test_accumulator(self):\n        loop = GameLoop(fixed_dt=0.1)\n        loop.tick(0.05, lambda dt: None, lambda: None)\n        assert abs(loop.accumulator - 0.05) < 1e-9\n\n    def test_max_steps(self):\n        loop = GameLoop(fixed_dt=0.01, max_steps=3)\n        count = [0]\n        loop.tick(10.0, lambda dt: count.__setitem__(0, count[0] + 1), lambda: None)\n        assert count[0] <= 3\n\n    def test_accumulator_reset(self):\n        loop = GameLoop(fixed_dt=0.01, max_steps=2)\n        loop.tick(1.0, lambda dt: None, lambda: None)\n        assert loop.accumulator == 0.0\n\n    def test_total_counts(self):\n        loop = GameLoop(fixed_dt=0.01)\n        loop.tick(0.05, lambda dt: None, lambda: None)\n        assert loop.total_updates == 5\n        assert loop.total_renders == 1\n\n    def test_max_frame_time(self):\n        loop = GameLoop(fixed_dt=0.01, max_steps=100, max_frame_time=0.1)\n        count = [0]\n        loop.tick(5.0, lambda dt: count.__setitem__(0, count[0] + 1), lambda: None)\n        assert count[0] <= 10\n''')

print("✅ tests/test_game_loop.py создан")

## 7.3. Тесты для компонентов и `Entity`

In [ ]:
with open('mystic_detective/tests/test_components.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"Тесты компонентов.\"\"\"\n\nfrom core.components import Position, Health, Inventory, MoveSpeed\n\n\nclass TestPosition:\n    def test_move(self):\n        p = Position(0, 0)\n        p.move(10, 20)\n        assert p.x == 10\n        assert p.y == 20\n\n    def test_distance(self):\n        a = Position(0, 0)\n        b = Position(3, 4)\n        assert a.distance_to(b) == 5.0\n\n\nclass TestHealth:\n    def test_damage(self):\n        h = Health(100, 100)\n        h.damage(30)\n        assert h.current == 70\n\n    def test_damage_below_zero(self):\n        h = Health(100, 100)\n        h.damage(999)\n        assert h.current == 0\n\n    def test_heal(self):\n        h = Health(50, 100)\n        h.heal(30)\n        assert h.current == 80\n\n    def test_heal_above_max(self):\n        h = Health(50, 100)\n        h.heal(999)\n        assert h.current == 100\n\n    def test_is_alive(self):\n        assert Health(1, 100).is_alive\n        assert not Health(0, 100).is_alive\n\n\nclass TestInventory:\n    def test_add(self):\n        inv = Inventory()\n        inv.add(\"sword\")\n        inv.add(\"sword\")\n        assert inv.size == 1\n\n    def test_remove(self):\n        inv = Inventory()\n        inv.add(\"sword\")\n        inv.remove(\"sword\")\n        assert inv.size == 0\n\n    def test_independent_lists(self):\n        a = Inventory()\n        b = Inventory()\n        a.add(\"x\")\n        assert b.size == 0\n''')

with open('mystic_detective/tests/test_entity.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"Тесты Entity.\"\"\"\n\nfrom core.entity import Entity\nfrom core.components import Position, Health\n\n\nclass TestEntity:\n    def test_add_and_get(self):\n        e = Entity(\"test\")\n        e.add_component(Position(1, 2))\n        p = e.get(Position)\n        assert p.x == 1 and p.y == 2\n\n    def test_get_missing(self):\n        e = Entity(\"test\")\n        assert e.get(Position) is None\n\n    def test_has(self):\n        e = Entity(\"test\")\n        e.add_component(Position())\n        assert e.has(Position)\n        assert not e.has(Health)\n\n    def test_remove(self):\n        e = Entity(\"test\")\n        e.add_component(Position())\n        e.remove(Position)\n        assert not e.has(Position)\n\n    def test_replace_component(self):\n        e = Entity(\"test\")\n        e.add_component(Position(1, 1))\n        e.add_component(Position(2, 2))\n        assert e.get(Position).x == 2\n''')

print("✅ tests/test_components.py, tests/test_entity.py созданы")

## 7.4. Тесты для систем

In [ ]:
with open('mystic_detective/tests/test_update_systems.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"Тесты систем обновления.\"\"\"\n\nfrom core.entity import Entity\nfrom core.components import Position, Health, MoveSpeed, Timer\nfrom game.update_systems.movement_system import MovementSystem\nfrom game.update_systems.timer_system import TimerSystem\nfrom game.update_systems.health_system import HealthSystem\n\n\ndef make_mover(name=\"m\", speed=100.0):\n    e = Entity(name)\n    e.add_component(Position(0, 0))\n    e.add_component(MoveSpeed(speed))\n    return e\n\n\nclass TestMovementSystem:\n    def test_moves_entities(self):\n        e = make_mover()\n        MovementSystem().update([e], 1.0)\n        assert e.get(Position).x == 100.0\n\n    def test_skips_dead(self):\n        e = make_mover()\n        e.add_component(Health(0, 100))\n        MovementSystem().update([e], 1.0)\n        assert e.get(Position).x == 0.0\n\n    def test_skips_static(self):\n        e = Entity(\"door\")\n        e.add_component(Position(5, 5))\n        MovementSystem().update([e], 1.0)\n        assert e.get(Position).x == 5\n\n\nclass TestTimerSystem:\n    def test_fires(self):\n        fired = []\n        e = Entity(\"t\")\n        e.add_component(Timer(\"t1\", duration=0.5, callback=lambda: fired.append(1)))\n        TimerSystem().update([e], 0.6)\n        assert fired == [1]\n\n    def test_does_not_double_fire(self):\n        fired = []\n        e = Entity(\"t\")\n        e.add_component(Timer(\"t1\", duration=0.1, callback=lambda: fired.append(1)))\n        TimerSystem().update([e], 1.0)\n        TimerSystem().update([e], 1.0)\n        assert fired == [1]\n\n\nclass TestHealthSystem:\n    def test_deactivates_dead(self):\n        e = Entity(\"dead\")\n        e.add_component(Health(0, 100))\n        HealthSystem().update([e], 0.1)\n        assert not e.active\n''')

print("✅ tests/test_update_systems.py создан")

## 7.5. Запуск всех тестов

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-v', '--tb=short'],
    cwd='mystic_detective',
    capture_output=True,
    text=True,
)
print(result.stdout[-3000:])  # последние 3000 символов
if result.stderr:
    print("STDERR:", result.stderr[-1000:])
print(f"Код выхода: {result.returncode}")

---
# Шаг 8. Документация

## 8.1. ADR-004

In [ ]:
import os
os.makedirs('mystic_detective/docs', exist_ok=True)

with open('mystic_detective/docs/adr_004_game_loop.md', 'w', encoding='utf-8') as f:
    f.write('''# ADR-004: Введение Game Loop и компонентов

## Контекст
После модуля 3 Game был тонким оркестратором систем команд, но:
- мир не обновлялся сам (не было update по времени);
- не было таймеров и фоновых событий;
- сущности были монолитными классами (комбинаторный взрыв наследования).

## Решение
Добавить:
- GameLoop с fixed timestep и max_steps (защита от спирали);
- компоненты Position, Health, Inventory, MoveSpeed, Timer;
- Entity — контейнер компонентов;
- MovementSystem, TimerSystem, HealthSystem;
- Game.update(dt) вызывает все update-системы.

## Рассмотренные альтернативы
1. Variable timestep — проще, но физика нестабильна.
2. Оставить монолитные классы — быстро, но не масштабируется.
3. ECS — мощнее, но избыточно для текстовой игры.

## Последствия
- ✅ Стабильная физика на любом железе.
- ✅ Таймеры и фоновые события стали возможны.
- ✅ Компоненты переиспользуются между сущностями.
- ⚠️ Больше классов и косвенности.
- ⚠️ Новичку сложнее проследить путь update.
''')

print("✅ docs/adr_004_game_loop.md создан")

## 8.2. UML модуля 4

In [ ]:
uml = '''```mermaid
classDiagram
    class GameLoop {
        -fixed_dt: float
        -max_steps: int
        -_accumulator: float
        +tick(frame_time, update, render)
        +accumulator float
        +total_updates int
    }
    class Entity {
        -_components: dict
        +add_component(c)
        +get(type)
        +has(type)
        +remove(type)
    }
    class Position {
        +x: float
        +y: float
        +move(dx, dy)
    }
    class Health {
        +current: int
        +maximum: int
        +damage(n)
        +heal(n)
    }
    class MoveSpeed {
        +speed: float
    }
    class Timer {
        +name: str
        +duration: float
        +fired: bool
    }
    class MovementSystem {
        +update(entities, dt)
    }
    class TimerSystem {
        +update(entities, dt)
    }
    class HealthSystem {
        +update(entities, dt)
    }
    Entity o-- Position
    Entity o-- Health
    Entity o-- MoveSpeed
    Entity o-- Timer
    MovementSystem ..> Position
    MovementSystem ..> MoveSpeed
    TimerSystem ..> Timer
    HealthSystem ..> Health
```
'''
with open('mystic_detective/docs/uml_module4.md', 'w', encoding='utf-8') as f:
    f.write(uml)

print("✅ docs/uml_module4.md создан")

## 8.3. Проверка PEP 8

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'pycodestyle',
     'core/game_loop.py', 'core/components.py', 'core/entity.py',
     'game/update_systems/', 'game/game_v4.py', 'game/factory_v4.py'],
    cwd='mystic_detective',
    capture_output=True,
    text=True,
)
if result.stdout:
    print("⚠️ Замечания PEP 8:")
    print(result.stdout)
else:
    print("✅ PEP 8: замечаний нет")

---
# Итоги

## Что мы сделали

| Инструмент | Проблема | Улучшение |
|---|---|---|
| **`GameLoop`** | Мир не обновлялся | Фиксированный шаг + accumulator |
| **Защита от спирали** | Возможное зависание | max_steps + сброс аккумулятора |
| **Компоненты** | Монолитные сущности | dataclass-компоненты |
| **`Entity`** | Нет контейнера | get/has/remove |
| **`MovementSystem`** | Логика движения в классах | Единая система |
| **`TimerSystem`** | Нет таймеров | События «через N секунд» |
| **`HealthSystem`** | Нет единой обработки смерти | Одно место |
| **`Game.update(dt)`** | Нет обновления | Вызов всех update-систем |

## Что изменилось в проекте

**До:**
- `Game.run()` ждёт ввода игрока;
- нет времени;
- монолитные сущности.

**После:**
- `GameLoop` с фиксированным `dt`;
- защита от спирали смерти;
- компоненты `Position`, `Health`, `Inventory`, `MoveSpeed`, `Timer`;
- `Entity` — контейнер;
- `MovementSystem`, `TimerSystem`, `HealthSystem`;
- таймеры работают, мир обновляется сам.

## Как это подготовит следующие модули

- **Модуль 5 (State, Strategy, Command):** состояния игры будут использовать игровой цикл для анимаций переходов; Strategy пригодится для поведения Кузи.
- **Модуль 6 (Observer, Factory):** события (найдена улика, таймер сработал) пойдут через Observer.
- **Модуль 7 (ECS):** компоненты превратятся в полноценный ECS; системы станут более специализированными.
- **Pygame-версия:** игровой цикл и компоненты напрямую перейдут в Pygame-рендеринг.

## Литература

**Базовая:**
- Nystrom R. «Game Programming Patterns»: [Game Loop](https://gameprogrammingpatterns.com/game-loop.html), [Update Method](https://gameprogrammingpatterns.com/update-method.html), [Component](https://gameprogrammingpatterns.com/component.html).
- Лутц М. «Изучаем Python», главы 26–27.

**Углублённая:**
- [Gaffer On Games: Fix Your Timestep!](https://gafferongames.com/post/fix_your_timestep/)
- [Gaffer On Games: Integration Basics](https://gafferongames.com/post/integration_basics/)
- [GitHub: ECS FAQ](https://github.com/SanderMertens/ecs-faq)
- [Adam Martin: Entity Systems](https://t-machine.org/index.php/2007/09/03/entity-systems-are-the-future-of-mmog-development-part-1/)
- [Real Python: Pygame Tutorial](https://realpython.com/pygame-a-primer/)

## Что дальше

В модуле 5 мы изучим:
- **State** — управление состояниями игры (меню, расследование, бой, диалог);
- **Strategy** — взаимозаменяемые алгоритмы (поведение Кузи, типы атак);
- **Command** — инкапсуляция действий с поддержкой undo.